# Semana 09 — Modelagem de Dados e SQL

**Curso:** Análise de Dados com Python — SENAI (Turma T5)
**UC (MSEP):** Manipulação de Dados com Python e SQL (150h) — Bloco 5, Semana 09

Na Semana 08 você criou sua primeira tabela real (`pedidos`) e escreveu SELECT, WHERE e GROUP BY. Esta semana você aprende a **juntar tabelas diferentes** com JOIN, e a **salvar consultas prontas** com VIEW — os dois recursos que transformam várias tabelas separadas numa análise só.

> Em cada tópico abaixo: **2 exemplos resolvidos** + **1 atividade prática** para você fazer sozinho(a).

### 🧭 De onde você vem: a Semana 08

Você já tem o PostgreSQL instalado, o banco `sabor_caseiro` criado, e a tabela `pedidos` populada com os 18 pedidos do Restaurante Sabor Caseiro. Você já sabe escrever `SELECT`, filtrar com `WHERE`, e resumir com `COUNT`/`SUM`/`AVG`/`GROUP BY`. Esta semana usa a MESMA tabela `pedidos` — e cria uma tabela nova, `produtos`, pra você praticar como duas tabelas se relacionam.

---
### 🟢 Abertura — Semana 09: Modelagem de Dados e SQL

**O que você vai aprender nesta semana:**
- Relacionar tabelas diferentes com INNER JOIN, combinando JOIN com WHERE e GROUP BY
- Salvar consultas prontas como VIEW, pra não reescrever a mesma consulta toda vez

### 📌 Antes de começar

Esta semana continua no VS Code local, conectado ao mesmo banco `sabor_caseiro` da Semana 08. Você usa a tabela `pedidos` (já existe) e cria uma tabela nova, `produtos`, com o preço e a categoria de cada item do cardápio — é essa segunda tabela que te dá o que precisa pra praticar JOIN.

---
## 1. Joins no PostgreSQL

Os JOINs no SQL têm como objetivo relacionar diferentes tabelas do banco de dados. Com eles, você consegue cruzar informações de tabelas separadas — o mesmo tipo de combinação que você já fazia com `.merge()` no Pandas, desde a Semana 05.

Para criar um JOIN, o primeiro passo é descobrir qual coluna as duas tabelas têm em comum — é através dela que o SQL sabe como cruzar os dados.

| Tipo de JOIN | O que faz |
|---|---|
| `INNER JOIN` | Traz só as linhas que têm correspondência nas DUAS tabelas |
| `LEFT JOIN` | Traz TODAS as linhas da tabela da esquerda, mesmo sem correspondência na direita |

Nesta semana você pratica o `INNER JOIN` — o mais usado no dia a dia; os outros tipos (LEFT, RIGHT, FULL) você aprofunda na Semana 10.

### 🔹 Exemplo 1 — Criando a tabela produtos e relacionando com pedidos

📖 **Antes do código:** o código cria a tabela `produtos` (com `item`, `categoria` e `preco`) e a popula com os 4 itens do cardápio do Sabor Caseiro. Depois, o `INNER JOIN` relaciona `pedidos` e `produtos` pela coluna que as duas têm em comum: `item`.

In [ ]:
try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost", dbname="sabor_caseiro", user="postgres", password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS produtos (
            item VARCHAR(100) PRIMARY KEY,
            categoria VARCHAR(50),
            preco NUMERIC(10,2)
        );
    """)
    cursor.execute("TRUNCATE TABLE produtos;")
    cursor.executemany(
        "INSERT INTO produtos (item, categoria, preco) VALUES (%s, %s, %s);",
        [
            ("Marmita Executiva", "Prato Principal", 32.90),
            ("Feijoada Completa", "Prato Principal", 45.00),
            ("Suco Natural", "Bebida", 9.50),
            ("Sobremesa do Dia", "Sobremesa", 12.00),
        ],
    )
    conexao.commit()
    print("Tabela produtos criada e populada!")

    cursor.execute("""
        SELECT pedidos.cliente, pedidos.item, produtos.categoria, produtos.preco
        FROM pedidos
        INNER JOIN produtos ON pedidos.item = produtos.item
        LIMIT 5;
    """)
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão.")
except psycopg2.OperationalError as erro:
    print(f"Erro real do Python: {erro}")

### 🔹 Exemplo 2 — JOIN combinado com WHERE

📖 **Antes do código:** um JOIN pode ser combinado com WHERE, exatamente como qualquer outra consulta — o WHERE filtra o resultado JÁ combinado pelas duas tabelas. Aqui, a consulta traz só os pedidos de itens da categoria "Prato Principal".

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        SELECT pedidos.cliente, pedidos.item, produtos.preco
        FROM pedidos
        INNER JOIN produtos ON pedidos.item = produtos.item
        WHERE produtos.categoria = 'Prato Principal'
        ORDER BY produtos.preco DESC;
    """)
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão — reinstale com %pip install psycopg2-binary.")
except psycopg2.OperationalError as erro:
    print("Ainda não conectou. Confira, nos Serviços do Windows, se 'postgresql-x64-...' está 'Em execução', e se host/usuário/senha/banco estão corretos.")
    print(f"Erro real do Python: {erro}")


### ✏️ Atividade Prática 1 — Sua vez de programar

**Contextualização:** a diretoria quer saber o faturamento total por categoria de produto, pra decidir onde investir em divulgação.

**Comando:** escreva uma consulta que faça INNER JOIN entre `pedidos` e `produtos`, agrupando por `produtos.categoria` e somando `pedidos.valor AS faturamento`, ordenada do maior faturamento para o menor.

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        SELECT produtos.categoria,
               SUM(pedidos.valor) AS faturamento
        FROM pedidos
        INNER JOIN produtos ON pedidos.item = produtos.item
        GROUP BY produtos.categoria
        ORDER BY faturamento DESC;
    """)
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão — reinstale com %pip install psycopg2-binary.")
except psycopg2.OperationalError as erro:
    print("Ainda não conectou. Confira, nos Serviços do Windows, se 'postgresql-x64-...' está 'Em execução', e se host/usuário/senha/banco estão corretos.")
    print(f"Erro real do Python: {erro}")


---
## 2. Views no PostgreSQL

Até aqui você viu como criar diferentes consultas aos bancos de dados. Mas onde ficam esses resultados? A resposta é: em lugar nenhum! Cada `SELECT` que você roda é perdido assim que você executa outro — nada fica salvo.

Uma **VIEW** resolve isso: ela salva uma consulta (não o resultado, a CONSULTA em si) com um nome, pra você não precisar reescrevê-la toda vez. Uma VIEW se comporta como uma tabela normal em qualquer `SELECT` posterior — só que, por trás, ela sempre roda a consulta original de novo, trazendo o dado atualizado.

### 🔹 Exemplo 1 — Criando uma VIEW

📖 **Antes do código:** `CREATE VIEW nome AS SELECT ...` salva a consulta com o nome escolhido. A partir daí, `SELECT * FROM nome` roda a consulta salva, como se fosse uma tabela.

In [ ]:
try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost", dbname="sabor_caseiro", user="postgres", password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()

    cursor.execute("""
        CREATE OR REPLACE VIEW vw_pedidos_completos AS
        SELECT pedidos.id_pedido, pedidos.cliente, pedidos.filial,
               produtos.item, produtos.categoria, produtos.preco
        FROM pedidos
        INNER JOIN produtos ON pedidos.item = produtos.item;
    """)
    conexao.commit()
    print("View vw_pedidos_completos criada!")

    cursor.execute("SELECT * FROM vw_pedidos_completos LIMIT 5;")
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão.")
except psycopg2.OperationalError as erro:
    print(f"Erro real do Python: {erro}")

### 🔹 Exemplo 2 — Consultando a VIEW com filtro

📖 **Antes do código:** uma VIEW aceita WHERE, ORDER BY e qualquer outro comando, exatamente como uma tabela normal — a única diferença é que ela não guarda dado, só a consulta.

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        SELECT cliente, item, preco
        FROM vw_pedidos_completos
        WHERE filial = 'Centro'
        ORDER BY preco DESC;
    """)
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão — reinstale com %pip install psycopg2-binary.")
except psycopg2.OperationalError as erro:
    print("Ainda não conectou. Confira, nos Serviços do Windows, se 'postgresql-x64-...' está 'Em execução', e se host/usuário/senha/banco estão corretos.")
    print(f"Erro real do Python: {erro}")


### ✏️ Atividade Prática 2 — Sua vez de programar

**Contextualização:** a equipe financeira quer uma consulta pronta que sempre mostre o faturamento por filial já com o nome da categoria — sem precisar reescrever o JOIN toda vez que for consultar.

**Comando:** crie uma VIEW chamada `vw_faturamento_filial` que traga `filial`, `categoria` e `SUM(preco) AS total`, agrupando por `filial` e `categoria` — depois, consulte a VIEW criada com um `SELECT *`.

In [ ]:
try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost", dbname="sabor_caseiro", user="postgres", password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()

    cursor.execute("""
        CREATE OR REPLACE VIEW vw_faturamento_filial AS
        SELECT pedidos.filial, produtos.categoria, SUM(produtos.preco) AS total
        FROM pedidos
        INNER JOIN produtos ON pedidos.item = produtos.item
        GROUP BY pedidos.filial, produtos.categoria;
    """)
    conexao.commit()
    print("View vw_faturamento_filial criada!")

    cursor.execute("SELECT * FROM vw_faturamento_filial ORDER BY total DESC;")
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão.")
except psycopg2.OperationalError as erro:
    print(f"Erro real do Python: {erro}")

---
## 3. Treino em Squads — Sexta-feira (Encontro 3)

Cada squad recebe 2 tabelas relacionadas e precisa escrever um INNER JOIN respondendo à pergunta do cenário.

### Squad B — Petshop

**Contextualização:** o petshop quer saber o nome do dono de cada pet, junto com o serviço agendado.

In [ ]:
try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost", dbname="sabor_caseiro", user="postgres", password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS squad_b_pets (pet VARCHAR(50) PRIMARY KEY, dono VARCHAR(100));
    """)
    cursor.execute("TRUNCATE TABLE squad_b_pets;")
    cursor.executemany(
        "INSERT INTO squad_b_pets (pet, dono) VALUES (%s, %s);",
        [("Rex", "Carla Nunes"), ("Mimi", "Pedro Alves")],
    )
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS squad_b_agendamentos (pet VARCHAR(50), servico VARCHAR(50));
    """)
    cursor.execute("TRUNCATE TABLE squad_b_agendamentos;")
    cursor.executemany(
        "INSERT INTO squad_b_agendamentos (pet, servico) VALUES (%s, %s);",
        [("Rex", "Banho e Tosa"), ("Mimi", "Vacina")],
    )
    conexao.commit()
    print("Tabelas do Squad B prontas!")

    # escreva aqui o INNER JOIN entre squad_b_pets e squad_b_agendamentos, pela coluna pet
    cursor.execute("""
        -- SUA CONSULTA AQUI
    """)
    print(cursor.fetchall())
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão.")
except psycopg2.OperationalError as erro:
    print(f"Erro real do Python: {erro}")

### Squad C — Biblioteca

**Contextualização:** a biblioteca quer saber o título de cada livro emprestado, junto com o nome do leitor.

In [ ]:
try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost", dbname="sabor_caseiro", user="postgres", password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS squad_c_livros (codigo VARCHAR(10) PRIMARY KEY, titulo VARCHAR(100));
    """)
    cursor.execute("TRUNCATE TABLE squad_c_livros;")
    cursor.executemany(
        "INSERT INTO squad_c_livros (codigo, titulo) VALUES (%s, %s);",
        [("L01", "Dom Casmurro"), ("L02", "O Cortiço")],
    )
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS squad_c_emprestimos (codigo VARCHAR(10), leitor VARCHAR(100));
    """)
    cursor.execute("TRUNCATE TABLE squad_c_emprestimos;")
    cursor.executemany(
        "INSERT INTO squad_c_emprestimos (codigo, leitor) VALUES (%s, %s);",
        [("L01", "Simone Prado"), ("L02", "André Lima")],
    )
    conexao.commit()
    print("Tabelas do Squad C prontas!")

    # escreva aqui o INNER JOIN entre squad_c_livros e squad_c_emprestimos, pela coluna codigo
    cursor.execute("""
        -- SUA CONSULTA AQUI
    """)
    print(cursor.fetchall())
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão.")
except psycopg2.OperationalError as erro:
    print(f"Erro real do Python: {erro}")

### 🗣️ Debate coletivo (após as apresentações)

- Algum squad tentou fazer o JOIN pela coluna errada? O que aconteceu com o resultado?
- Alguém percebeu que, sem o ON certo, o JOIN traria linhas sem sentido (produto cartesiano)?
- Como uma VIEW ajudaria o squad a não reescrever esse JOIN toda vez?

---
### 🏁 Fechamento — Semana 09

**Nesta semana você aprendeu:**
- Relacionar tabelas diferentes com INNER JOIN
- Combinar JOIN com WHERE e GROUP BY
- Salvar consultas prontas com VIEW

**Próxima semana:** a Semana 10 aprofunda o SQL — Operações CRUD completas (INSERT, UPDATE, DELETE), funções de número/texto/data, subconsultas, e uma primeira olhada em variáveis, functions e transactions no PostgreSQL.

---
### Assinatura

Curso: **Análise de Dados com Python — SENAI (Turma T5)**
Semana 09 — Modelagem de Dados e SQL

*Prof. Especialista Cláudio F. Neves*